In [20]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from BaselineRemoval import BaselineRemoval
import matplotlib.pyplot as plt
import glob

In [21]:
def modified_z_score(intensity):
    median_int = np.median(intensity)
    mad_int = np.median([np.abs(intensity - median_int)])
    modified_z_scores = 0.6745 * (intensity - median_int) / mad_int
    return modified_z_scores

In [22]:
def fixer(y,m):
    threshold = 5 # binarization threshold. 
    spikes = abs(np.array(modified_z_score(np.diff(y)))) > threshold
    y_out = y.copy() # So we don’t overwrite y
    for i in np.arange(len(spikes)):
        if spikes[i] != 0: # If we have an spike in position i
            w = np.arange(i-m,i+1+m) # we select 2 m + 1 points around our spike
            w2 = w[spikes[w] == 0] # From such interval, we choose the ones which are not spikes
            y_out[i] = np.mean(y[w2])  # and we average their values
    return y_out

In [27]:
def preprocess_data(data_path, save_path, data_delim="\t"):  
    l = [pd.read_csv(filename,header=None,delimiter = data_delim) for filename in glob.glob(data_path)]

    data = pd.concat(l, ignore_index=True)
    data.columns = ["positionX", "positionY", "Wavenumber", "Counts"]
    data.drop(data.columns[[0, 1]], axis = 1, inplace = True)

    n =1300
    data_separate = pd.DataFrame(np.hstack(np.vsplit(data.values, len(data) // n)))
    data_separate.columns = data_separate.columns.map(lambda c: chr(c + ord('A')))

    col_to_drop = data_separate.columns[np.array([i for i in range(data_separate.shape[1]) if i != 0 and i%2 != 1])]

    rslt_df = data_separate.drop(col_to_drop, axis=1)
    
    rslt_df.rename(columns = {'A':'Wavenumber'}, inplace = True)

    # Transform the data to a numpy array
    #data1 = rslt_df.drop(columns = ['Wavenumber'])
    MASTER = []
    for i in rslt_df.columns:
        intensity = rslt_df[i].values.tolist()
        intensity = np.array(intensity)
        fixed_intensity = fixer(intensity,m=7)  
        MASTER.append(fixed_intensity)
        
    df = pd.DataFrame(MASTER)
    result = df.transpose()
    result.columns = result.columns.map(lambda c: chr(c + ord('A')))
    #result.to_excel('M1 cosmic ray removal.xlsx', index=False)

    #baseline removal 

    data1 = result.drop(columns = ['A'])

    cols = list(data1.columns.values)
    #ind = data. iloc[:, 0]
    #ind_list = ind.tolist()

    Master2=[]

    for i in data1.columns:
        temp = data1[i].values.tolist()
        baseObj=BaselineRemoval(temp)
        Zhangfit_output=baseObj.ZhangFit()
        Master2.append(Zhangfit_output)
        
    output = pd.DataFrame (Master2)

    reshape = np.transpose(output)

    #reshape.to_excel('M1 baseline correction.xlsx', index=False)

    #Normalization

    x = reshape.values #returns a numpy array
    min_max_scaler = preprocessing.MinMaxScaler()
    x_scaled = min_max_scaler.fit_transform(x)
    data_normalized = pd.DataFrame(x_scaled, columns = [cols])

    #output
    new_df = rslt_df['Wavenumber']
    a = new_df.to_frame(name='Wavenumber')

    frame = pd.concat([a,data_normalized], axis = 1)
    frame.to_excel(save_path, index=False)
    return frame

In [28]:
data_path = './ML_demo_data/Control/*.txt'
data_delim = '\t'
save_path = './ML_demo_data/Control/processed_data.xlsx'

In [29]:
# preprocess_data(data_path, save_path)

In [30]:
data_folders = ["0.2mg", "4mg", "10mg", "Control"]

combined = pd.DataFrame()

for folder in data_folders:
    data_path = f'./ML_demo_data/{folder}/*.txt'
    save_path = f'./ML_demo_data/{folder}/{folder}- No Processing.xlsx'
    frame = preprocess_data(data_path, save_path)
    frame = frame.T
    if folder == "0.2mg":
        new_cols = frame.loc['Wavenumber'].to_list()
    new_index = [f'{folder}' for i in range(frame.shape[0])]
    frame.insert(0, 'Group', new_index)
    frame.reset_index(drop=True, inplace=True)
    frame = frame[1:]
    combined = pd.concat([combined, frame], axis=0, ignore_index=True)

In [32]:
combined.to_excel("./Combined_Preprocessing.xlsx", index=False)

In [31]:
combined

,Group,0,1,2,3,4,5,6,7,8,...,1290,1291,1292,1293,1294,1295,1296,1297,1298,1299
0,0.2mg,0.091960,0.201028,0.110428,0.130370,0.177970,0.206065,0.201635,0.177668,0.217175,...,0.106479,0.037792,0.062685,0.071094,0.099895,0.051015,0.079565,0.089713,0.106002,0.095704
1,0.2mg,0.120056,0.119188,0.084778,0.098544,0.122716,0.111779,0.141167,0.121121,0.144012,...,0.087302,0.071383,0.060302,0.083368,0.072116,0.052638,0.077048,0.060582,0.079926,0.115641
2,0.2mg,0.076114,0.111441,0.174717,0.190455,0.152493,0.188189,0.171704,0.227370,0.100323,...,0.071654,0.058284,0.067917,0.056314,0.083135,0.109963,0.051922,0.070849,0.066625,0.124128
3,0.2mg,0.081976,0.131748,0.111894,0.157274,0.079930,0.181326,0.097806,0.138612,0.139015,...,0.066752,0.094492,0.094909,0.099237,0.076225,0.109842,0.057525,0.092943,0.042394,0.091176
4,0.2mg,0.103629,0.120111,0.110325,0.119700,0.170386,0.142894,0.121300,0.120356,0.148933,...,0.107301,0.115345,0.082621,0.098078,0.043098,0.115679,0.091855,0.144073,0.059027,0.079525
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,Control,0.047940,0.064891,0.075027,0.054443,0.048717,0.085628,0.036270,0.056325,0.038685,...,0.041223,0.051628,0.045845,0.035077,0.050450,0.042153,0.037591,0.033028,0.047126,0.003867
237,Control,0.123874,0.073619,0.082340,0.084696,0.107733,0.046480,0.080464,0.090601,0.118239,...,0.055305,0.075780,0.072289,0.082785,0.101272,0.059809,0.090253,0.078723,0.067186,0.029642
238,Control,0.056023,0.063127,0.083276,0.046834,0.057055,0.058342,0.089435,0.075822,0.067172,...,0.044774,0.048959,0.034429,0.032314,0.023873,0.071438,0.044100,0.069204,0.025618,0.049325
239,Control,0.066879,0.089949,0.060887,0.088054,0.078167,0.128353,0.066048,0.139274,0.114039,...,0.035945,0.039567,0.087949,0.083351,0.128543,0.067723,0.056687,0.124376,0.077965,0.058868
